# 🤖 Day 18 — RLHF, Reward Modelling, PPO & LLM Fine-Tuning Patterns
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | RLHF & Reward Modelling (Bradley-Terry) | ~12 sec |
| 2 | 10:30–11:00 AM | PPO Concepts in Tabular RL | ~5 sec |
| 3 | 11:00 AM–1:00 PM | LLM Fine-Tuning Patterns (LoRA, DPO) | ~18 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | RLHF Capstone: Full 5-Stage Pipeline | ~20 sec |

> **Zero downloads. Pure numpy + sklearn + scipy.**
---

In [ ]:
# !pip install scikit-learn scipy matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit as sigmoid
from scipy.optimize import minimize
from scipy import stats
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import sklearn
print('All imports OK')
print(f'   sklearn : {sklearn.__version__}')

---
## Session 1 — RLHF & Reward Modelling
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

### RLHF pipeline
1. **SFT**: supervised fine-tuning on demonstration data
2. **RM**: train reward model on pairwise human preferences
3. **RL**: optimise policy against reward model using PPO


In [ ]:
# 1.1  Bradley-Terry Reward Model from Scratch
# Bradley-Terry model: P(i preferred over j) = σ(r_i - r_j)
# Train by maximising log-likelihood over observed preference pairs

rng = np.random.default_rng(42)
N_ITEMS = 12   # 12 candidate responses to rank

# Simulate true latent reward scores (unknown to the model)
true_rewards = rng.normal(0, 1, N_ITEMS)
print('True reward scores (hidden):')
for i, r in enumerate(true_rewards):
    print(f'  Item {i:2d}: {r:+.4f}')

# Generate synthetic pairwise preferences
# Human labeller chooses preferred item with some noise
N_PAIRS = 200
pairs = [(rng.integers(N_ITEMS), rng.integers(N_ITEMS))
         for _ in range(N_PAIRS * 2)]
pairs = [(i, j) for i, j in pairs if i != j][:N_PAIRS]

# Stochastic preference: higher reward wins with prob σ(r_i - r_j)
preferences = []
for i, j in pairs:
    p_ij = sigmoid(true_rewards[i] - true_rewards[j])
    winner = i if rng.random() < p_ij else j
    loser  = j if winner == i else i
    preferences.append((winner, loser))

print(f'\nGenerated {len(preferences)} pairwise preferences')
print(f'Example: Item {preferences[0][0]} preferred over Item {preferences[0][1]}')

In [ ]:
# 1.2  Train Bradley-Terry Reward Model via MLE
def bradley_terry_nll(rewards, preferences):
    """Negative log-likelihood: -sum log σ(r_winner - r_loser)."""
    nll = 0.0
    for winner, loser in preferences:
        prob = sigmoid(rewards[winner] - rewards[loser])
        nll -= np.log(prob + 1e-10)
    return nll

def bradley_terry_gradient(rewards, preferences):
    """Gradient of NLL w.r.t. reward scores."""
    grad = np.zeros(len(rewards))
    for winner, loser in preferences:
        prob = sigmoid(rewards[winner] - rewards[loser])
        # d(-log σ(rw-rl))/drw = -(1-σ) = σ-1
        grad[winner] += prob - 1
        grad[loser]  += 1 - prob
    return grad

# Optimise with L-BFGS-B
result = minimize(
    fun     = lambda r: bradley_terry_nll(r, preferences),
    jac     = lambda r: bradley_terry_gradient(r, preferences),
    x0      = np.zeros(N_ITEMS),
    method  = 'L-BFGS-B',
    options = {'maxiter': 500}
)

learned_rewards = result.x
# Normalise for comparison
learned_rewards -= learned_rewards.mean()
true_rewards_norm = true_rewards - true_rewards.mean()

spearman_r = stats.spearmanr(true_rewards, learned_rewards).correlation
print(f'Bradley-Terry optimisation: success={result.success}  NLL={result.fun:.4f}')
print(f'Spearman rank correlation with true rewards: {spearman_r:.4f}')

In [ ]:
# 1.3  Reward Model Evaluation: Preference Prediction Accuracy
def preference_accuracy(rewards, preferences):
    """What fraction of preferences does the reward model predict correctly?"""
    correct = 0
    for winner, loser in preferences:
        if rewards[winner] > rewards[loser]:
            correct += 1
    return correct / len(preferences)

acc_learned = preference_accuracy(learned_rewards, preferences)
acc_random  = preference_accuracy(rng.normal(0,1,N_ITEMS), preferences)

print(f'Reward model preference accuracy : {acc_learned:.4f}')
print(f'Random baseline accuracy         : {acc_random:.4f}')

# Visualise learned vs true rewards
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(true_rewards_norm, learned_rewards, color='#534AB7', s=80, alpha=0.8)
for i in range(N_ITEMS):
    axes[0].annotate(str(i), (true_rewards_norm[i], learned_rewards[i]),
                     xytext=(3,3), textcoords='offset points', fontsize=8)
axes[0].plot([-2,2],[-2,2],'k--',alpha=0.4)
axes[0].set_xlabel('True reward (normalised)')
axes[0].set_ylabel('Learned reward (Bradley-Terry)')
axes[0].set_title(f'Reward Correlation (Spearman r={spearman_r:.3f})')
axes[0].grid(alpha=0.3)

# Preference prediction probabilities
all_probs = [sigmoid(learned_rewards[w] - learned_rewards[l]) for w,l in preferences]
axes[1].hist(all_probs, bins=20, color='#1D9E75', alpha=0.85, edgecolor='white')
axes[1].axvline(0.5, color='#D85A30', linestyle='--', linewidth=2, label='Random (0.5)')
axes[1].set_xlabel('P(winner > loser) from reward model')
axes[1].set_ylabel('Count')
axes[1].set_title('Reward Model Confidence Distribution')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 1.4  Elo Rating System (Simplified Reward Model)
class EloRatingSystem:
    def __init__(self, n_items, initial_rating=1500, K=32):
        self.ratings = np.full(n_items, float(initial_rating))
        self.K       = K
        self.history = [self.ratings.copy()]

    def expected_score(self, ra, rb):
        return 1 / (1 + 10**((rb - ra) / 400))

    def update(self, winner, loser):
        E_w = self.expected_score(self.ratings[winner], self.ratings[loser])
        self.ratings[winner] += self.K * (1 - E_w)
        self.ratings[loser]  += self.K * (0 - (1 - E_w))
        self.history.append(self.ratings.copy())

elo = EloRatingSystem(N_ITEMS)
for winner, loser in preferences:
    elo.update(winner, loser)

elo_spearman = stats.spearmanr(true_rewards, elo.ratings).correlation
print('Elo Ratings after all matches:')
ranked_idx = np.argsort(elo.ratings)[::-1]
for rank, idx in enumerate(ranked_idx[:5], 1):
    print(f'  Rank {rank}: Item {idx:2d}  Elo={elo.ratings[idx]:.0f}  true_r={true_rewards[idx]:+.3f}')
print(f'\nElo Spearman correlation: {elo_spearman:.4f}  (Bradley-Terry: {spearman_r:.4f})')

---
## Session 2 — PPO Concepts in Tabular RL
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  REINFORCE Policy Gradient (Bandit Setting)
rng = np.random.default_rng(42)

# 5-arm bandit with known true rewards
TRUE_REWARDS = np.array([0.2, 0.6, 0.4, 0.8, 0.3])
N_ACTIONS    = len(TRUE_REWARDS)
OPTIMAL_ACTION = TRUE_REWARDS.argmax()

logits   = np.zeros(N_ACTIONS)   # policy parameters
LR       = 0.1
N_EPISODES = 400

def policy_probs(logits):
    shifted = logits - logits.max()
    e = np.exp(shifted)
    return e / e.sum()

reward_history   = []
action_history   = []
baseline_history = []

for ep in range(N_EPISODES):
    probs    = policy_probs(logits)
    action   = rng.choice(N_ACTIONS, p=probs)
    reward   = TRUE_REWARDS[action] + rng.normal(0, 0.1)  # noisy reward
    baseline = probs @ TRUE_REWARDS                        # expected reward
    advantage = reward - baseline

    # Policy gradient: ∇log π(a|s) * advantage
    grad         = -probs.copy()
    grad[action] += 1
    logits       += LR * advantage * grad

    reward_history.append(reward)
    action_history.append(action)
    baseline_history.append(baseline)

final_probs = policy_probs(logits)
print('REINFORCE Policy after training:')
for a, (p, tr) in enumerate(zip(final_probs, TRUE_REWARDS)):
    bar = '█' * int(p * 20)
    mark = ' ← optimal' if a == OPTIMAL_ACTION else ''
    print(f'  Action {a}: prob={p:.4f}  true_r={tr:.1f}  {bar}{mark}')
print(f'\nOptimal action selected: {logits.argmax() == OPTIMAL_ACTION}')

In [ ]:
# 2.2  PPO Clipped Objective
def ppo_clipped_objective(advantage, prob_ratio, epsilon=0.2):
    """
    PPO clip: min(r_t * A_t, clip(r_t, 1-ε, 1+ε) * A_t)
    where r_t = π_θ(a|s) / π_θ_old(a|s)
    """
    clipped = np.clip(prob_ratio, 1 - epsilon, 1 + epsilon)
    return np.minimum(prob_ratio * advantage, clipped * advantage)

# Simulate different probability ratios
ratios     = np.linspace(0.5, 1.5, 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for advantage, color, label in [(1.0, '#1D9E75','A>0 (good action)'),
                                  (-1.0,'#D85A30','A<0 (bad action)')]:
    obj_ppo  = ppo_clipped_objective(advantage, ratios)
    obj_vpg  = ratios * advantage  # vanilla policy gradient (no clipping)
    axes[0].plot(ratios, obj_vpg,  linestyle='--', color=color, alpha=0.5, linewidth=1.5)
    axes[0].plot(ratios, obj_ppo,  linestyle='-',  color=color, linewidth=2.5, label=f'PPO {label}')

axes[0].axvline(1.0, color='black', linestyle=':', alpha=0.5)
axes[0].axvline(1.2, color='gray',  linestyle=':', alpha=0.4, label='Clip at 1+ε=1.2')
axes[0].axvline(0.8, color='gray',  linestyle=':', alpha=0.4, label='Clip at 1-ε=0.8')
axes[0].set_xlabel('Probability ratio r_t = π_new / π_old')
axes[0].set_ylabel('Objective value')
axes[0].set_title('PPO Clipped Objective (ε=0.2)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# PPO vs REINFORCE training comparison
reward_window = pd.Series(reward_history).rolling(20).mean()
axes[1].plot(reward_window, color='#534AB7', linewidth=1.8, label='REINFORCE')
axes[1].axhline(TRUE_REWARDS.max(), linestyle='--', color='#D85A30',
                linewidth=1.5, label=f'Optimal reward={TRUE_REWARDS.max()}')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Reward (rolling mean)')
axes[1].set_title('REINFORCE Convergence')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 2.3  Entropy Bonus + KL Divergence Monitor
def policy_entropy(probs):
    return -np.sum(probs * np.log(probs + 1e-9))

def kl_divergence(p_old, p_new):
    return np.sum(p_old * np.log((p_old + 1e-9) / (p_new + 1e-9)))

# Track entropy across training
logits_track = np.zeros(N_ACTIONS)
entropies    = []
kl_divs      = []

for ep in range(200):
    probs_old  = policy_probs(logits_track)
    action     = rng.choice(N_ACTIONS, p=probs_old)
    reward     = TRUE_REWARDS[action] + rng.normal(0, 0.1)
    advantage  = reward - (probs_old @ TRUE_REWARDS)
    grad       = -probs_old.copy(); grad[action] += 1
    logits_track += 0.05 * advantage * grad
    probs_new  = policy_probs(logits_track)
    entropies.append(policy_entropy(probs_new))
    kl_divs.append(kl_divergence(probs_old, probs_new))

print(f'Initial entropy : {entropies[0]:.4f}  (log({N_ACTIONS})={np.log(N_ACTIONS):.4f} max)')
print(f'Final entropy   : {entropies[-1]:.4f}  (lower = more deterministic)')
print(f'Mean KL per step: {np.mean(kl_divs):.6f}  (PPO target: < 0.015)')

---
## Session 3 — LLM Fine-Tuning Patterns
**Time:** 11:00 AM–1:00 PM | **Run time: ~18 sec**

In [ ]:
# 3.1  LoRA: Low-Rank Adaptation
rng = np.random.default_rng(42)

class LoRALayer:
    """
    LoRA wraps a frozen weight matrix W and adds a trainable low-rank update:
    W_eff = W_frozen + alpha/rank * A @ B
    """
    def __init__(self, d_in, d_out, rank=4, alpha=16, seed=42):
        rng_l       = np.random.default_rng(seed)
        self.W      = rng_l.normal(0, 0.02, (d_in, d_out))   # frozen
        self.A      = rng_l.normal(0, 0.01, (d_in, rank))    # trainable
        self.B      = np.zeros((rank, d_out))                  # init to 0
        self.rank   = rank
        self.alpha  = alpha
        self.scale  = alpha / rank

    def forward(self, x):
        base_out = x @ self.W
        lora_out = x @ self.A @ self.B * self.scale
        return base_out + lora_out

    def n_trainable(self):
        return self.A.size + self.B.size

    def n_total(self):
        return self.W.size + self.A.size + self.B.size

# Compare parameter counts for different ranks
D_IN, D_OUT = 768, 768  # typical transformer hidden size
print('LoRA Parameter Efficiency:')
print(f'{"Rank":8s} | {"LoRA params":12s} | {"Full params":12s} | {"% of full"}')
print('-' * 50)
for rank in [1, 2, 4, 8, 16, 32]:
    lora = LoRALayer(D_IN, D_OUT, rank=rank)
    pct  = lora.n_trainable() / lora.n_total() * 100
    print(f'{rank:8d} | {lora.n_trainable():12,} | {lora.n_total():12,} | {pct:.2f}%')

In [ ]:
# 3.2  Simulate LoRA Fine-Tuning
def lora_forward(X, W_frozen, A, B, scale):
    return X @ W_frozen + (X @ A) @ B * scale

def lora_update(X, y_target, W_frozen, A, B, scale, lr=0.001):
    """Simple gradient step on A and B only (W_frozen is fixed)."""
    h_a  = X @ A                          # (n, rank)
    h_ab = h_a @ B                        # (n, d_out)
    y_pred = X @ W_frozen + h_ab * scale  # forward
    err    = y_pred - y_target            # residual

    # Gradients
    dB = h_a.T @ err * scale / len(X)    # (rank, d_out)
    dA = X.T @ (err @ B.T * scale) / len(X)  # (d_in, rank)

    # Update only A and B
    A -= lr * dA
    B -= lr * dB
    loss = np.mean(err**2)
    return A, B, loss

# Simulate regression with LoRA fine-tuning
D = 32; RANK = 4
rng_l = np.random.default_rng(42)
W_frozen = rng_l.normal(0, 0.1, (D, D))
A_lora   = rng_l.normal(0, 0.01, (D, RANK))
B_lora   = np.zeros((RANK, D))
SCALE    = 16.0 / RANK

# Target: simulate a task the frozen model can't do
X_lora   = rng_l.normal(0, 1, (200, D))
y_target = np.sin(X_lora @ rng_l.normal(0, 0.1, D)).reshape(-1, 1) * np.ones((1, D))

losses_lora = []
for step in range(100):
    A_lora, B_lora, loss = lora_update(X_lora, y_target, W_frozen, A_lora, B_lora, SCALE)
    losses_lora.append(loss)

print(f'LoRA fine-tuning (rank={RANK}):')
print(f'  Initial loss: {losses_lora[0]:.6f}')
print(f'  Final loss  : {losses_lora[-1]:.6f}')
print(f'  Params trained: {A_lora.size + B_lora.size} / {W_frozen.size + A_lora.size + B_lora.size}')

In [ ]:
# 3.3  Direct Preference Optimisation (DPO) Loss
def dpo_loss(log_pi_w, log_pi_l, log_pi_ref_w, log_pi_ref_l, beta=0.1):
    """
    DPO loss for a batch of preference pairs (w=winner, l=loser):
    L_DPO = -log σ(β * (log π_θ(w|x) - log π_ref(w|x))
                   - β * (log π_θ(l|x) - log π_ref(l|x)))
    Removes need for explicit reward model.
    """
    rewards_w = beta * (log_pi_w - log_pi_ref_w)  # implicit reward for winner
    rewards_l = beta * (log_pi_l - log_pi_ref_l)  # implicit reward for loser
    loss = -np.mean(np.log(sigmoid(rewards_w - rewards_l) + 1e-9))
    return loss

# Simulate DPO training dynamics
rng_dpo = np.random.default_rng(42)
N_DPO   = 64
BETA    = 0.1

dpo_losses = []
# Simulate: reference policy assigns uniform log-probs
log_pi_ref_w = np.full(N_DPO, -1.0)
log_pi_ref_l = np.full(N_DPO, -1.0)

for step in range(50):
    # Policy gradually assigns higher logprob to winners
    noise = rng_dpo.normal(0, 0.05, N_DPO)
    log_pi_w = -1.0 + step * 0.02 + noise       # improving on winners
    log_pi_l = -1.0 - step * 0.01 + noise * 0.5  # degrading on losers
    loss = dpo_loss(log_pi_w, log_pi_l, log_pi_ref_w, log_pi_ref_l, BETA)
    dpo_losses.append(loss)

print(f'DPO training simulation:')
print(f'  Initial loss: {dpo_losses[0]:.4f}')
print(f'  Final loss  : {dpo_losses[-1]:.4f}')
print(f'  Improvement : {(dpo_losses[0]-dpo_losses[-1])/dpo_losses[0]:.1%}')

In [ ]:
# 3.4  DPO vs PPO Conceptual Comparison + Soft Prompt Tuning
print('=== DPO vs PPO Comparison ===')
comparison = {
    'Requires reward model'    : ('Yes (explicit RM)',   'No (implicit in loss)'),
    'Requires RL training loop': ('Yes (on-policy PPO)', 'No (supervised loss)'),
    'Training stability'       : ('Lower (reward hacking)','Higher (stable CE loss)'),
    'Computational cost'       : ('High (RL overhead)',  'Low (single forward pass)'),
    'Flexibility'              : ('High (complex reward)','Medium (pairwise only)'),
    'Alignment quality'        : ('State-of-art (RLHF)', 'Comparable on many tasks'),
}
for aspect, (ppo_val, dpo_val) in comparison.items():
    print(f'  {aspect:30s}: PPO={ppo_val:30s} | DPO={dpo_val}')

# Soft Prompt Tuning concept
print('\n=== Soft Prompt Tuning ===')
SEQ_LEN, D_MODEL = 10, 64
N_SOFT_TOKENS    = 5      # prepend 5 learnable tokens

rng_sp     = np.random.default_rng(42)
soft_prompt = rng_sp.normal(0, 0.01, (N_SOFT_TOKENS, D_MODEL))  # trainable
X_tokens   = rng_sp.normal(0, 1, (SEQ_LEN, D_MODEL))            # frozen token embeds
X_combined = np.vstack([soft_prompt, X_tokens])

print(f'Original sequence length : {SEQ_LEN} tokens × {D_MODEL}D')
print(f'After soft prefix        : {SEQ_LEN + N_SOFT_TOKENS} tokens × {D_MODEL}D')
print(f'Trainable parameters     : {soft_prompt.size} (prompt only)')
print(f'Frozen parameters        : {X_tokens.size} (model weights fixed)')

---
## Lunch Break — 1:00–2:00 PM
1. What are the 3 stages of the RLHF pipeline, in order?
2. Why does LoRA initialise B to zero at the start of training?
3. How does DPO avoid needing a separate reward model?
4. Why does PPO clip the probability ratio to [1-ε, 1+ε]?
---

## Session 5 — RLHF Capstone: Full 5-Stage Pipeline
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

In [ ]:
# RLHF Pipeline on Tabular Data (Breast Cancer as proxy)
# Each 'response' = a feature subset prediction; 'human preference' = accuracy
print('='*60)
print(' RLHF CAPSTONE: 5-STAGE ALIGNMENT PIPELINE')
print('='*60)

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)
scaler_bc = StandardScaler().fit(X_tr)
X_tr_sc   = scaler_bc.transform(X_tr)
X_te_sc   = scaler_bc.transform(X_te)

# Stage 1: SFT — Supervised Fine-Tuning Baseline
print('\nStage 1: SFT (Supervised Fine-Tuning)')
sft_model = LogisticRegression(max_iter=500).fit(X_tr_sc, y_tr)
sft_acc   = sft_model.score(X_te_sc, y_te)
sft_probs = sft_model.predict_proba(X_te_sc)
print(f'  SFT accuracy: {sft_acc:.4f}')
print(f'  SFT mean confidence: {sft_probs.max(axis=1).mean():.4f}')

In [ ]:
# Stage 2: Collect Pairwise Preferences from Simulated Human Labellers
print('\nStage 2: Preference Data Collection')
rng_cap = np.random.default_rng(0)

# Generate 5 candidate 'model variants' with different regularisation
models_pool = [
    LogisticRegression(C=c, max_iter=500).fit(X_tr_sc, y_tr)
    for c in [0.01, 0.1, 1.0, 10.0, 100.0]
]
model_accs  = [m.score(X_te_sc, y_te) for m in models_pool]
model_probs = [m.predict_proba(X_te_sc) for m in models_pool]

# 'Human preference': prefer model with higher confidence on correct predictions
def human_preference_score(model, X_te, y_te):
    """Simulated human: prefers model that is confident AND correct."""
    probs  = model.predict_proba(X_te)
    preds  = probs.argmax(axis=1)
    correct_conf = probs[np.arange(len(y_te)), y_te][preds == y_te]
    return correct_conf.mean() if len(correct_conf) > 0 else 0.0

human_scores = [human_preference_score(m, X_te_sc, y_te) for m in models_pool]

print(f'  Model pool ({len(models_pool)} candidates):')
for i, (acc, hs) in enumerate(zip(model_accs, human_scores)):
    print(f'    Model {i} (C=10^{i-2}): acc={acc:.4f}  human_score={hs:.4f}')

# Generate pairwise preferences
N_PREF = 150
pref_pairs = [(rng_cap.integers(5), rng_cap.integers(5)) for _ in range(N_PREF*2)]
pref_pairs = [(i,j) for i,j in pref_pairs if i!=j][:N_PREF]
human_prefs = []
for i, j in pref_pairs:
    p_ij = sigmoid(human_scores[i] - human_scores[j])
    winner = i if rng_cap.random() < p_ij else j
    loser  = j if winner == i else i
    human_prefs.append((winner, loser))

print(f'  Generated {len(human_prefs)} preference pairs')

In [ ]:
# Stage 3: Train Bradley-Terry Reward Model
print('\nStage 3: Reward Model Training (Bradley-Terry)')

rm_result = minimize(
    fun     = lambda r: -sum(np.log(sigmoid(r[w]-r[l])+1e-9) for w,l in human_prefs),
    jac     = lambda r: np.array([sum(sigmoid(r[w]-r[l])-1 if i==w else
                                      1-sigmoid(r[w]-r[l]) if i==l else 0
                                      for w,l in human_prefs) for i in range(5)]),
    x0      = np.zeros(5),
    method  = 'L-BFGS-B'
)
rm_scores    = rm_result.x
rm_spearman  = stats.spearmanr(human_scores, rm_scores).correlation
rm_pref_acc  = sum(1 for w,l in human_prefs if rm_scores[w]>rm_scores[l]) / len(human_prefs)

print(f'  Reward model Spearman r: {rm_spearman:.4f}')
print(f'  Preference accuracy    : {rm_pref_acc:.4f}')
print(f'  Learned reward scores  : {rm_scores.round(4)}')

In [ ]:
# Stage 4: Policy Optimisation via REINFORCE + Reward Model
print('\nStage 4: RL Policy Optimisation (REINFORCE guided by RM)')

N_ACTIONS_rl = 5  # which model variant to deploy
logits_rl    = np.zeros(N_ACTIONS_rl)
LR_RL        = 0.15
rl_rewards   = []

for ep in range(300):
    probs_rl = policy_probs(logits_rl)
    action   = rng_cap.choice(N_ACTIONS_rl, p=probs_rl)
    # Reward = reward model score + small noise
    reward   = rm_scores[action] + rng_cap.normal(0, 0.05)
    baseline = probs_rl @ rm_scores
    advantage = reward - baseline
    grad      = -probs_rl.copy(); grad[action] += 1
    logits_rl += LR_RL * advantage * grad
    rl_rewards.append(reward)

best_policy_action = logits_rl.argmax()
print(f'  Policy converged to: Model {best_policy_action} (C=10^{best_policy_action-2})')
print(f'  Best model accuracy: {model_accs[best_policy_action]:.4f}')
print(f'  Human score        : {human_scores[best_policy_action]:.4f}')

In [ ]:
# Stage 5: DPO Alternative — Direct Preference Optimisation
print('\nStage 5: DPO Alternative (no explicit reward model)')

# DPO: train policy directly on preference pairs
dpo_logits  = np.zeros(N_ACTIONS_rl)  # policy parameters
BETA_DPO    = 0.5
LR_DPO      = 0.1
dpo_loss_hist = []

for step in range(200):
    probs_theta = policy_probs(dpo_logits)
    probs_ref   = np.ones(N_ACTIONS_rl) / N_ACTIONS_rl  # uniform reference

    total_grad  = np.zeros(N_ACTIONS_rl)
    total_loss  = 0.0
    for w, l in human_prefs[:50]:  # mini-batch
        log_ratio_w = np.log(probs_theta[w]+1e-9) - np.log(probs_ref[w]+1e-9)
        log_ratio_l = np.log(probs_theta[l]+1e-9) - np.log(probs_ref[l]+1e-9)
        delta   = BETA_DPO * (log_ratio_w - log_ratio_l)
        sig_val = sigmoid(delta)
        loss_i  = -np.log(sig_val + 1e-9)
        total_loss += loss_i
        # Gradient on logits_dpo
        coef    = -(1 - sig_val) * BETA_DPO
        total_grad[w] += coef * (1 - probs_theta[w])
        total_grad[l] -= coef * (1 - probs_theta[l])

    dpo_logits  -= LR_DPO * total_grad / 50
    dpo_loss_hist.append(total_loss / 50)

best_dpo_action = dpo_logits.argmax()
print(f'  DPO converged to   : Model {best_dpo_action}')
print(f'  DPO model accuracy : {model_accs[best_dpo_action]:.4f}')
print(f'  PPO model accuracy : {model_accs[best_policy_action]:.4f}')
print(f'  DPO initial loss   : {dpo_loss_hist[0]:.4f}')
print(f'  DPO final loss     : {dpo_loss_hist[-1]:.4f}')

In [ ]:
# Final Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reward model vs human scores
axes[0].scatter(human_scores, rm_scores, color='#534AB7', s=100, zorder=5)
for i, (h,r) in enumerate(zip(human_scores, rm_scores)):
    axes[0].annotate(f'M{i}', (h, r), xytext=(4,4), textcoords='offset points', fontsize=9)
axes[0].set_xlabel('Human preference score'); axes[0].set_ylabel('Learned reward (BT)')
axes[0].set_title(f'Reward Model vs Human (r={rm_spearman:.3f})')
axes[0].grid(alpha=0.3)

# Policy convergence (REINFORCE)
rl_smooth = pd.Series(rl_rewards).rolling(30).mean()
axes[1].plot(rl_smooth, color='#534AB7', linewidth=2, label='REINFORCE')
axes[1].axhline(rm_scores.max(), linestyle='--', color='#D85A30', label='Max RM score')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Reward')
axes[1].set_title('Stage 4: RL Policy Convergence')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# DPO loss curve
axes[2].plot(dpo_loss_hist, color='#1D9E75', linewidth=2)
axes[2].set_xlabel('Step'); axes[2].set_ylabel('DPO loss')
axes[2].set_title('Stage 5: DPO Training Loss')
axes[2].grid(alpha=0.3)

plt.suptitle('RLHF Capstone: 5-Stage Alignment Pipeline', fontsize=12)
plt.tight_layout(); plt.show()

print('\n=== FINAL RLHF PIPELINE REPORT ===')
print(f'  SFT baseline accuracy   : {sft_acc:.4f}')
print(f'  RM preference accuracy  : {rm_pref_acc:.4f}')
print(f'  PPO best model accuracy : {model_accs[best_policy_action]:.4f}')
print(f'  DPO best model accuracy : {model_accs[best_dpo_action]:.4f}')

---
## Day 18 Summary

| Concept | Skill Gained |
|---|---|
| RLHF pipeline | 3 stages: SFT → RM → RL fine-tuning |
| Bradley-Terry | P(i>j) = σ(r_i - r_j), MLE training on preference pairs |
| Elo rating | Iterative K-factor update, alternative to BT |
| REINFORCE | Policy gradient with advantage = reward - baseline |
| PPO clip | Constrain policy update via [1-ε, 1+ε] ratio clipping |
| Entropy bonus | Encourage exploration; track KL from reference |
| LoRA | Frozen W + trainable A@B, rank parameter efficiency |
| DPO loss | Implicit reward without RM: log σ(β*(log ratios)) |
| Soft prompt tuning | Learnable prefix tokens prepended to frozen model |
| 5-stage RLHF | SFT → prefs → BT reward → REINFORCE → DPO comparison |

---
## Day 19 Preview
- Retrieval-Augmented Generation (RAG) deep dive
- Vector databases: FAISS-style indexing and ANN search
- Long-context summarisation strategies
- Tool-use and function-calling patterns
- Capstone: build a production RAG Q&A system

In [ ]:
# Bonus: Day 18 in one cell
rng_b = np.random.default_rng(0)
true_r_b = rng_b.normal(0,1,6)
prefs_b = [(i,j) if true_r_b[i]>true_r_b[j] else (j,i)
            for i,j in [(rng_b.integers(6),rng_b.integers(6)) for _ in range(50)] if i!=j]
res_b = minimize(lambda r:-sum(np.log(sigmoid(r[w]-r[l])+1e-9) for w,l in prefs_b),
                 np.zeros(6), method='L-BFGS-B')
print(f'BT spearman: {stats.spearmanr(true_r_b,res_b.x).correlation:.4f}')
print(f'PPO optimal action: {logits_rl.argmax()} (RL aligned to human preferences)')
print(f'DPO final loss: {dpo_loss_hist[-1]:.4f}')
print('\nDay 18 complete — RLHF, Bradley-Terry, PPO, LoRA, DPO, and 5-stage alignment!')